You can download the `requirements.txt` for this course from the workspace of this lab. `File --> Open...`

# L2: Create Agents to Research and Write an Article

In this lesson, you will be introduced to the foundational concepts of multi-agent systems and get an overview of the crewAI framework.

The libraries are already installed in the classroom. If you're running this notebook on your own machine, you can install the following:
```Python
!pip install crewai==0.28.8 crewai_tools==0.1.6 langchain_community==0.0.29
```

In [1]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

- Import from the crewAI libray.

In [5]:
from crewai import Agent, Task, Crew

- As a LLM for your agents, you'll be using OpenAI's `gpt-3.5-turbo`.

**Optional Note:** crewAI also allow other popular models to be used as a LLM for your Agents. You can see some of the examples at the [bottom of the notebook](#1).

In [ ]:
import os
from utils import get_openai_api_key

openai_api_key = get_openai_api_key()
os.environ["OPENAI_MODEL_NAME"] = 'gpt-3.5-turbo'

## Creating Agents

- Define your Agents, and provide them a `role`, `goal` and `backstory`.
- It has been seen that LLMs perform better when they are role playing.

### Agent: Planner

**Note**: The benefit of using _multiple strings_ :
```Python
varname = "line 1 of text"
          "line 2 of text"
```

versus the _triple quote docstring_:
```Python
varname = """line 1 of text
             line 2 of text
          """
```
is that it can avoid adding those whitespaces and newline characters, making it better formatted to be passed to the LLM.

In [ ]:
planner = Agent(
    role="Content Planner",
    goal="Plan engaging and factually accurate content on {topic}",
    backstory="You're working on planning a blog article "
              "about the topic: {topic}."
              "You collect information that helps the "
              "audience learn something "
              "and make informed decisions. "
              "Your work is the basis for "
              "the Content Writer to write an article on this topic.",
    allow_delegation=False,
	verbose=True
)

### Agent: Writer

In [ ]:
writer = Agent(
    role="Content Writer",
    goal="Write insightful and factually accurate "
         "opinion piece about the topic: {topic}",
    backstory="You're working on a writing "
              "a new opinion piece about the topic: {topic}. "
              "You base your writing on the work of "
              "the Content Planner, who provides an outline "
              "and relevant context about the topic. "
              "You follow the main objectives and "
              "direction of the outline, "
              "as provide by the Content Planner. "
              "You also provide objective and impartial insights "
              "and back them up with information "
              "provide by the Content Planner. "
              "You acknowledge in your opinion piece "
              "when your statements are opinions "
              "as opposed to objective statements.",
    allow_delegation=False,
    verbose=True
)

### Agent: Editor

In [ ]:
editor = Agent(
    role="Editor",
    goal="Edit a given blog post to align with "
         "the writing style of the organization. ",
    backstory="You are an editor who receives a blog post "
              "from the Content Writer. "
              "Your goal is to review the blog post "
              "to ensure that it follows journalistic best practices,"
              "provides balanced viewpoints "
              "when providing opinions or assertions, "
              "and also avoids major controversial topics "
              "or opinions when possible.",
    allow_delegation=False,
    verbose=True
)

## Creating Tasks

- Define your Tasks, and provide them a `description`, `expected_output` and `agent`.

### Task: Plan

In [ ]:
plan = Task(
    description=(
        "1. Prioritize the latest trends, key players, "
            "and noteworthy news on {topic}.\n"
        "2. Identify the target audience, considering "
            "their interests and pain points.\n"
        "3. Develop a detailed content outline including "
            "an introduction, key points, and a call to action.\n"
        "4. Include SEO keywords and relevant data or sources."
    ),
    expected_output="A comprehensive content plan document "
        "with an outline, audience analysis, "
        "SEO keywords, and resources.",
    agent=planner,
)

### Task: Write

In [ ]:
write = Task(
    description=(
        "1. Use the content plan to craft a compelling "
            "blog post on {topic}.\n"
        "2. Incorporate SEO keywords naturally.\n"
		"3. Sections/Subtitles are properly named "
            "in an engaging manner.\n"
        "4. Ensure the post is structured with an "
            "engaging introduction, insightful body, "
            "and a summarizing conclusion.\n"
        "5. Proofread for grammatical errors and "
            "alignment with the brand's voice.\n"
    ),
    expected_output="A well-written blog post "
        "in markdown format, ready for publication, "
        "each section should have 2 or 3 paragraphs.",
    agent=writer,
)

### Task: Edit

In [ ]:
edit = Task(
    description=("Proofread the given blog post for "
                 "grammatical errors and "
                 "alignment with the brand's voice."),
    expected_output="A well-written blog post in markdown format, "
                    "ready for publication, "
                    "each section should have 2 or 3 paragraphs.",
    agent=editor
)

## Creating the Crew

- Create your crew of Agents
- Pass the tasks to be performed by those agents.
    - **Note**: *For this simple example*, the tasks will be performed sequentially (i.e they are dependent on each other), so the _order_ of the task in the list _matters_.
- `verbose=2` allows you to see all the logs of the execution. 

In [ ]:
crew = Crew(
    agents=[planner, writer, editor],
    tasks=[plan, write, edit],
    verbose=2
)

## Running the Crew

**Note**: LLMs can provide different outputs for they same input, so what you get might be different than what you see in the video.

In [ ]:
result = crew.kickoff(inputs={"topic": "Artificial Intelligence"})

 [DEBUG]: == Working Agent: Content Planner
 [INFO]: == Starting Task: 1. Prioritize the latest trends, key players, and noteworthy news on Artificial Intelligence.
2. Identify the target audience, considering their interests and pain points.
3. Develop a detailed content outline including an introduction, key points, and a call to action.
4. Include SEO keywords and relevant data or sources.


> Entering new CrewAgentExecutor chain...
I now can give a great answer

Final Answer: 

Title: The Future of Artificial Intelligence: Latest Trends and Key Players

Introduction:
- Brief overview of Artificial Intelligence and its impact on various industries
- Mention of recent advancements in AI technology

Key Points:
1. Latest Trends in Artificial Intelligence
- Machine learning algorithms
- Natural language processing
- Computer vision
- Autonomous vehicles
- Robotics
- AI ethics and regulation

2. Key Players in the AI Industry
- Google DeepMind
- IBM Watson
- Amazon Web Services
- Micros

I now can give a great answer

Final Answer: 

# The Future of Artificial Intelligence: Latest Trends and Key Players

## Introduction
Artificial Intelligence (AI) has been revolutionizing various industries with its ability to automate tasks, analyze data, and make decisions with minimal human intervention. Recent advancements in AI technology have propelled innovation across sectors, leading to improved efficiency, accuracy, and productivity. As AI continues to evolve, it is essential for individuals to stay informed about the latest trends and key players in the AI industry to understand its potential impact on society.

## Latest Trends in Artificial Intelligence
Machine learning algorithms have been a driving force behind AI advancements, enabling systems to learn from data and improve over time. Natural language processing has made significant strides in understanding and generating human language, opening up new possibilities for communication and interaction. Computer vision ha

- Display the results of your execution as markdown in the notebook.

In [ ]:
from IPython.display import Markdown
Markdown(result)

# The Future of Artificial Intelligence: Latest Trends and Key Players

## Introduction
Artificial Intelligence (AI) has been revolutionizing various industries with its ability to automate tasks, analyze data, and make decisions with minimal human intervention. Recent advancements in AI technology have propelled innovation across sectors, leading to improved efficiency, accuracy, and productivity. As AI continues to evolve, it is essential for individuals to stay informed about the latest trends and key players in the AI industry to understand its potential impact on society.

## Latest Trends in Artificial Intelligence
Machine learning algorithms have been a driving force behind AI advancements, enabling systems to learn from data and improve over time. Natural language processing has made significant strides in understanding and generating human language, opening up new possibilities for communication and interaction. Computer vision has enhanced the capabilities of AI systems to interpret and analyze visual information, leading to applications in image recognition, surveillance, and autonomous vehicles. Speaking of autonomous vehicles, AI has played a crucial role in developing self-driving cars, making transportation safer and more efficient. Robotics has also benefited from AI technology, with robots being used in manufacturing, healthcare, and other industries to perform complex tasks with precision. Discussions around AI ethics and regulation have become increasingly important to ensure responsible development and deployment of AI systems.

## Key Players in the AI Industry
Several companies have emerged as key players in the AI industry, driving innovation and shaping the future of AI technology. Google DeepMind has made significant contributions to AI research, particularly in the areas of deep learning and reinforcement learning. IBM Watson is known for its cognitive computing capabilities, helping businesses leverage AI for data analysis and decision-making. Amazon Web Services offers a range of AI services and tools for developers, making it easier to integrate AI into applications. Microsoft Azure provides cloud-based AI solutions, empowering organizations to build and deploy AI models at scale. NVIDIA is a leader in AI hardware, designing GPUs that accelerate AI workloads and training models. OpenAI is a research organization focused on advancing AI in a safe and beneficial way, promoting collaboration and transparency in AI development.

## Noteworthy News in Artificial Intelligence
Recent breakthroughs in AI research have led to advancements in natural language processing, computer vision, and robotics, opening up new possibilities for AI applications in healthcare, finance, and other sectors. However, challenges and controversies surrounding AI development, such as bias in AI algorithms and concerns about job displacement, highlight the need for ethical considerations and regulatory frameworks to guide the responsible use of AI. It is crucial for individuals to engage in discussions about the ethical implications of AI technology and stay informed about the latest news on AI developments to contribute to the responsible and sustainable growth of AI in the digital age.

## Conclusion
As AI continues to shape the future of technology and innovation, individuals from various backgrounds must explore the advancements in machine learning, natural language processing, computer vision, autonomous vehicles, robotics, AI ethics, and regulation to gain valuable insights into the evolving landscape of artificial intelligence. By fostering a deeper understanding of AI trends and key players, individuals can play a role in ensuring the responsible and sustainable growth of AI in society. Staying informed, discussing ethical implications, and keeping up with the latest news on AI developments are crucial steps towards contributing to the positive impact of AI technology in the digital age.

## Try it Yourself

- Pass in a topic of your choice and see what the agents come up with!

In [ ]:
topic = "Big Data"
result = crew.kickoff(inputs={"topic": topic})

 [DEBUG]: == Working Agent: Content Planner
 [INFO]: == Starting Task: 1. Prioritize the latest trends, key players, and noteworthy news on Big Data.
2. Identify the target audience, considering their interests and pain points.
3. Develop a detailed content outline including an introduction, key points, and a call to action.
4. Include SEO keywords and relevant data or sources.


> Entering new CrewAgentExecutor chain...
I now can give a great answer

Final Answer: 

Content Plan: 
Title: Unleashing the Power of Big Data: Trends, Players, and News

Outline:
I. Introduction
    A. Definition of Big Data
    B. Importance of Big Data in today's digital age
II. Latest Trends in Big Data
    A. Artificial Intelligence and Machine Learning
    B. Edge Computing
    C. Data Privacy and Security
III. Key Players in the Big Data Industry
    A. Google
    B. Amazon Web Services
    C. Microsoft
IV. Noteworthy News in Big Data
    A. Recent advancements in data analytics
    B. Impact of Big Da

I now can give a great answer

Final Answer:
# Unleashing the Power of Big Data: Trends, Players, and News

In today's digital age, the term "Big Data" has become increasingly prevalent, shaping the way businesses operate and make decisions. Big Data refers to the vast volume of structured and unstructured data that organizations can potentially mine for insights and information. The importance of Big Data lies in its ability to uncover patterns, trends, and associations that can lead to better decision-making and strategic planning.

One of the most significant trends in Big Data is the integration of Artificial Intelligence (AI) and Machine Learning (ML) technologies. These advancements allow for more accurate predictions and insights from data analysis. Additionally, Edge Computing has emerged as a key trend, enabling data processing and analysis to occur closer to the source of the data, reducing latency and improving efficiency. Data Privacy and Security have also become critical 

In [ ]:
Markdown(result)

# Unleashing the Power of Big Data: Trends, Players, and News

In today's digital age, the term "Big Data" has become increasingly prevalent, shaping the way businesses operate and make decisions. Big Data refers to the vast volume of structured and unstructured data that organizations can potentially mine for insights and information. The importance of Big Data lies in its ability to uncover patterns, trends, and associations that can lead to better decision-making and strategic planning.

One of the most significant trends in Big Data is the integration of Artificial Intelligence (AI) and Machine Learning (ML) technologies. These advancements allow for more accurate predictions and insights from data analysis. Additionally, Edge Computing has emerged as a key trend, enabling data processing and analysis to occur closer to the source of the data, reducing latency and improving efficiency. Data Privacy and Security have also become critical concerns in the Big Data landscape, with regulations like GDPR driving organizations to prioritize the protection of sensitive information.

Several tech giants have established themselves as key players in the Big Data industry. Google, with its Cloud Platform and tools like Google BigQuery, offers robust data analytics capabilities. Amazon Web Services (AWS) provides a range of Big Data services through its Amazon EMR and Amazon Redshift platforms. Microsoft Azure has also made significant strides in the Big Data space, offering services such as Azure Data Lake and Azure HDInsight.

Recent advancements in data analytics have revolutionized the way organizations extract insights from their data. From predictive analytics to real-time data processing, these innovations have empowered businesses to make data-driven decisions with accuracy and speed. The impact of Big Data on the healthcare industry has been particularly profound, with data analytics enabling personalized medicine, disease prediction, and improved patient outcomes.

The blog post on Big Data is tailored to a diverse audience, including data scientists and analysts seeking to expand their knowledge, business executives and decision-makers looking to leverage Big Data for strategic advantage, technology enthusiasts interested in the latest trends, and students and researchers exploring the potential of Big Data for future innovations.

To enhance the visibility of the blog post, key SEO keywords such as "Big Data trends," "key players in Big Data industry," and "latest news in Big Data" have been strategically incorporated throughout the content.

For further reading and exploration of Big Data trends, key players, and news, readers are encouraged to refer to reputable sources such as Gartner's Big Data trends report, Forbes articles on Big Data, and Harvard Business Review case studies on Big Data.

In conclusion, the blog post aims to inform and engage readers on the advancements and significance of Big Data in today's digital landscape. By staying updated on the latest trends and insights in Big Data, readers can harness the power of data analytics to drive innovation and success in their respective fields. Don't miss out on the opportunity to unlock the potential of Big Data. Stay informed, stay ahead!

<a name='1'></a>
 ## Other Popular Models as LLM for your Agents

#### Hugging Face (HuggingFaceHub endpoint)

```Python
from langchain_community.llms import HuggingFaceHub

llm = HuggingFaceHub(
    repo_id="HuggingFaceH4/zephyr-7b-beta",
    huggingfacehub_api_token="<HF_TOKEN_HERE>",
    task="text-generation",
)

### you will pass "llm" to your agent function
```

#### Mistral API

```Python
OPENAI_API_KEY=your-mistral-api-key
OPENAI_API_BASE=https://api.mistral.ai/v1
OPENAI_MODEL_NAME="mistral-small"
```

#### Cohere

```Python
from langchain_community.chat_models import ChatCohere
# Initialize language model
os.environ["COHERE_API_KEY"] = "your-cohere-api-key"
llm = ChatCohere()

### you will pass "llm" to your agent function
```

### For using Llama locally with Ollama and more, checkout the crewAI documentation on [Connecting to any LLM](https://docs.crewai.com/how-to/LLM-Connections/).

CrewAI supports all langchain tools